In [33]:
from langchain_mistralai import ChatMistralAI
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

from dotenv import load_dotenv
load_dotenv()

True

In [34]:
sub_llm = ChatMistralAI()

In [35]:
class SubState(TypedDict):
    eng_input: str
    hing_output: str

In [36]:
def translate_in_hindi(state: SubState):
    query = f"translate the following english text into hinglish text, Do not add any extra information keep the context clear text:{state['eng_input']}"
    answer = sub_llm.invoke(query).content
    return {"hing_output": answer}

In [37]:
sub_builder = StateGraph(SubState)

sub_builder.add_node("translate_in_hindi", translate_in_hindi)

sub_builder.add_edge(START, "translate_in_hindi")
sub_builder.add_edge("translate_in_hindi", END)

sub_graph = sub_builder.compile()

In [38]:
# Parent Graph
class ParentState(TypedDict):
    question: str
    english_answer: str
    hinglish_answer: str

llm = ChatMistralAI()

In [39]:
def generate_answer(state: ParentState):
    question = state['question']
    response = llm.invoke(question).content
    return {"english_answer" : response}

In [40]:
def translate_answer(state: ParentState):
    english_question = state['english_answer']
    response = sub_graph.invoke({
        "eng_input": english_question
    })
    return {"hinglish_answer": response["hing_output"]}

In [41]:
parent_builder = StateGraph(ParentState)

parent_builder.add_node("generate_answer", generate_answer)
parent_builder.add_node("translate_answer", translate_answer)

parent_builder.add_edge(START, "generate_answer")
parent_builder.add_edge("generate_answer", "translate_answer")
parent_builder.add_edge("translate_answer", END)

chatbot = parent_builder.compile()

result = chatbot.invoke({
    "question": "what is deep learning"
})
print(result["hinglish_answer"])

Here's the Hinglish translation of your text while keeping the context clear and not adding any extra information:

---

**Deep Learning (DL)** ek **machine learning (ML)** ka hissa hai jo **artificial neural networks (ANNs)** ka istemaal karta hai jismein **bahut saare layers** hote hain (isliye "deep" kehte hain) taaki complex problems ko model aur solve kiya ja sake. Yeh **human brain ke structure aur function** se inspire hai, khas taur par kaise **neurons** information ko process aur transmit karte hain.

---

### **Deep Learning ki Key Features:**
1. **Bahut saare layers wale Neural Networks** – Traditional ML ke uttar mein, deep learning models mein **10+ hidden layers** hote hain, jisse woh data ke hierarchical representations seekh sakte hain.
2. **Automatic Feature Extraction** – Deep learning models **khud hi** raw data se relevant features seekh lete hain (jaise images ke pixels, text ke words) bina manual engineering ke.
3. **Bade Data par Nirbhar** – Effective training ke